# 推荐模型（2017-2020 综合交互权重版）

本 Notebook 使用 `feature_engineering.ipynb` 生成的特征产物：

1. `book_tfidf_matrix.npz`：图书内容 TF-IDF 矩阵。
2. `books_info.pkl`：图书元数据映射表。
3. `user_item_weights.pkl`：用户-图书交互权重矩阵。

`INTEREST_WEIGHT` 当前采用“弱化时间”综合交互价值：

- 时间衰减权重：0.15
- 题名 TF-IDF 均值：0.30
- 主题词 TF-IDF 均值：0.25
- 图书总被借阅频次：0.20
- 用户总借阅频次：0.10

推荐模型仍采用混合策略：`Hybrid Score = α * Content-Based + (1 - α) * Collaborative Filtering`。


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity


def resolve_feature_dir():
    candidates = [Path('data/features'), Path('../data/features')]
    for p in candidates:
        if (p / 'book_tfidf_matrix.npz').exists() and (p / 'books_info.pkl').exists() and (p / 'user_item_weights.pkl').exists():
            return p
    raise FileNotFoundError('未找到 data/features 或 ../data/features 下的特征工程产物')


features_dir = resolve_feature_dir()
print('features_dir =', features_dir)


In [ ]:
class CampusHybridRecommender:
    def __init__(self, hybrid_weight=0.6, n_components=50):
        self.hybrid_weight = hybrid_weight
        self.n_components = n_components
        self.load_data()
        self.build_cf_model()

    def load_data(self):
        print('正在加载特征工程数据...')
        self.tfidf_matrix = sp.load_npz(features_dir / 'book_tfidf_matrix.npz')
        self.books_info = pd.read_pickle(features_dir / 'books_info.pkl')
        self.ui_weights = pd.read_pickle(features_dir / 'user_item_weights.pkl')

        self.books_info['TITLE'] = self.books_info['TITLE'].fillna('')
        self.book_id_to_idx = {book_id: idx for idx, book_id in enumerate(self.books_info['BOOK_ID'])}
        self.idx_to_book_id = {idx: book_id for book_id, idx in self.book_id_to_idx.items()}

        self.unique_users = self.ui_weights['USERID'].unique()
        self.user_id_to_idx = {user_id: idx for idx, user_id in enumerate(self.unique_users)}

        self.unique_titles = self.books_info['TITLE'].drop_duplicates().tolist()
        self.title_to_idx = {title: idx for idx, title in enumerate(self.unique_titles)}
        self.book_idx_to_title_idx = np.array([self.title_to_idx[title] for title in self.books_info['TITLE']])

        book_id_to_title = dict(zip(self.books_info['BOOK_ID'], self.books_info['TITLE']))
        self.ui_weights['TITLE_IDX'] = self.ui_weights['BOOK_ID'].map(lambda x: self.title_to_idx.get(book_id_to_title.get(x, ''), 0))

        print(f'数据加载完毕: {len(self.unique_users)} 名用户, {len(self.books_info)} 本图书, {len(self.unique_titles)} 个唯一书名。')
        print('INTEREST_WEIGHT 描述统计:')
        print(self.ui_weights['INTEREST_WEIGHT'].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).to_string())

    def build_cf_model(self):
        print('正在构建协同过滤矩阵分解模型...')
        row_indices = [self.user_id_to_idx[u] for u in self.ui_weights['USERID']]
        col_indices = self.ui_weights['TITLE_IDX'].tolist()
        data = self.ui_weights['INTEREST_WEIGHT'].astype(float)

        self.ui_matrix = sp.csr_matrix(
            (data, (row_indices, col_indices)),
            shape=(len(self.unique_users), len(self.unique_titles))
        )

        max_components = max(1, min(self.ui_matrix.shape) - 1)
        n_components = min(self.n_components, max_components)
        self.svd = TruncatedSVD(n_components=n_components, random_state=42)
        self.user_factors = self.svd.fit_transform(self.ui_matrix)
        self.item_factors = self.svd.components_.T
        print(f'协同过滤模型训练完成: n_components={n_components}, explained_variance={self.svd.explained_variance_ratio_.sum():.4f}')

    def get_content_based_scores(self, target_user_id):
        user_history = self.ui_weights[self.ui_weights['USERID'] == target_user_id]
        if user_history.empty:
            return np.zeros(len(self.books_info))

        top_history = user_history.sort_values(by='INTEREST_WEIGHT', ascending=False).head(3)
        book_indices = [self.book_id_to_idx[b] for b in top_history['BOOK_ID'] if b in self.book_id_to_idx]
        if not book_indices:
            return np.zeros(len(self.books_info))

        user_profile_vector = np.asarray(self.tfidf_matrix[book_indices].mean(axis=0))
        return cosine_similarity(user_profile_vector, self.tfidf_matrix).flatten()

    def get_collaborative_scores(self, target_user_id):
        if target_user_id not in self.user_id_to_idx:
            return np.zeros(len(self.books_info))

        u_idx = self.user_id_to_idx[target_user_id]
        cf_title_scores = np.dot(self.user_factors[u_idx], self.item_factors.T)
        cf_scores = cf_title_scores[self.book_idx_to_title_idx]

        if cf_scores.max() > cf_scores.min():
            cf_scores = (cf_scores - cf_scores.min()) / (cf_scores.max() - cf_scores.min())
        return cf_scores

    def recommend(self, target_user_id, top_n=10):
        if target_user_id not in self.user_id_to_idx:
            return []

        cb_scores = self.get_content_based_scores(target_user_id)
        cf_scores = self.get_collaborative_scores(target_user_id)
        hybrid_scores = self.hybrid_weight * cb_scores + (1 - self.hybrid_weight) * cf_scores

        user_history_ids = self.ui_weights[self.ui_weights['USERID'] == target_user_id]['BOOK_ID'].tolist()
        history_indices = [self.book_id_to_idx[b] for b in user_history_ids if b in self.book_id_to_idx]
        history_titles = set(self.books_info.iloc[history_indices]['TITLE'].tolist())

        recommended_indices = []
        seen_titles = set(history_titles)
        for idx in np.argsort(hybrid_scores)[::-1]:
            title = self.books_info.iloc[idx]['TITLE']
            if title and title not in seen_titles:
                recommended_indices.append(idx)
                seen_titles.add(title)
            if len(recommended_indices) >= top_n:
                break

        return pd.DataFrame([
            {
                'rank': rank + 1,
                'BOOK_ID': self.idx_to_book_id[idx],
                'TITLE': self.books_info.iloc[idx]['TITLE'],
                'hybrid_score': round(float(hybrid_scores[idx]), 6),
                'CALL_NO': self.books_info.iloc[idx].get('CALL_NO', ''),
            }
            for rank, idx in enumerate(recommended_indices)
        ])


In [ ]:
system = CampusHybridRecommender(hybrid_weight=0.6, n_components=50)

sample_user = system.ui_weights.groupby('USERID')['BOOK_ID'].count().sort_values(ascending=False).index[0]
print('sample_user =', sample_user)
system.recommend(sample_user, top_n=10)
